# FinQA hard-label BGE-small training on Colab GPU

This notebook builds the one shared training-row artifact from the validated Qwen cache, trains only the hard-label treatment, evaluates development only, and stores resumable checkpoints in Google Drive.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import os
import subprocess

drive.mount('/content/drive')
REPOSITORY = 'WatermelonfromEarth/finevid-distill'
PROJECT_DIR = Path('/content/finevid-distill')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/FinEvid-Distill')
TEACHER_DIR = ARTIFACT_ROOT / 'teacher_scores'
PROCESSED_DIR = ARTIFACT_ROOT / 'processed_data'
CHECKPOINT_DIR = ARTIFACT_ROOT / 'checkpoints' / 'hard_label_student'
for directory in (TEACHER_DIR, PROCESSED_DIR, CHECKPOINT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

token = userdata.get('GITHUB_TOKEN')
assert token, 'Add a read-only GITHUB_TOKEN in Colab Secrets before continuing.'
repository_url = f'https://github.com/{REPOSITORY}.git'
askpass = Path('/content/finevid-git-askpass.sh')
askpass.write_text("#!/bin/sh\ncase \"$1\" in\n  *Username*) printf '%s\\n' 'x-access-token' ;;\n  *) printf '%s\\n' \"$GITHUB_TOKEN\" ;;\nesac\n")
askpass.chmod(0o700)
git_environment = os.environ.copy()
git_environment.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': token})
def run_git(*arguments):
    result = subprocess.run(['git', *arguments], env=git_environment, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError((result.stderr or result.stdout).replace(token, '[REDACTED]'))
if PROJECT_DIR.exists():
    run_git('-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', 'main')
else:
    run_git('clone', '--depth', '1', repository_url, str(PROJECT_DIR))
run_git('-C', str(PROJECT_DIR), 'remote', 'set-url', 'origin', repository_url)
askpass.unlink(missing_ok=True)
del token, git_environment
print('Code:', PROJECT_DIR)
print('Persistent artifacts:', ARTIFACT_ROOT)

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU before continuing.'
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)

## Install and test without replacing Colab's CUDA-enabled PyTorch

In [ ]:
%pip install -q -r {PROJECT_DIR / 'requirements-colab.txt'}
%pip install -q -e {PROJECT_DIR} --no-deps

In [ ]:
import importlib
import sys

source_directory = str(PROJECT_DIR / 'src')
sys.path_importer_cache.pop(source_directory, None)
sys.path.insert(0, source_directory)
importlib.invalidate_caches()

def run_project(*arguments):
    return subprocess.run([sys.executable, *arguments], cwd=PROJECT_DIR, check=True)

run_project('-m', 'pytest', 'tests/test_training_rows.py', 'tests/test_hard_label_training.py', '-q')

## Materialize the shared training rows

This CPU-fast step validates the complete teacher cache, includes every gold fact, samples seeded within-report negatives, and writes the exact file that both student treatments must consume.

In [ ]:
TRAIN_ROWS = PROCESSED_DIR / 'train_rows.jsonl'
run_project(
    'src/data/build_training_rows.py',
    '--teacher-cache', str(TEACHER_DIR / 'teacher_train_scores.jsonl'),
    '--output', str(TRAIN_ROWS),
)

## Train or resume the hard-label student

The script automatically resumes only when a complete latest-checkpoint pointer exists. It never reads the test split.

In [ ]:
arguments = [
    'src/training/train_hard_labels.py',
    '--train-rows', str(TRAIN_ROWS),
    '--dev-data', str(PROJECT_DIR / 'data/processed/dev.jsonl'),
    '--output-dir', str(CHECKPOINT_DIR),
    '--device', 'cuda',
    '--epochs', '3',
    '--questions-per-batch', '8',
    '--mixed-precision', 'fp16',
]
if (CHECKPOINT_DIR / 'latest.json').exists():
    arguments.append('--resume')
run_project(*arguments)

## Confirm the persistent best checkpoint

In [ ]:
import json

verification = json.loads((CHECKPOINT_DIR / 'best_reload_verification.json').read_text())
assert verification['reload_verified'] is True
print(json.dumps(verification, indent=2))